In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from anthropic import Anthropic
client = Anthropic()

In [20]:
def llm(prompt):
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.content[0].text

In [21]:
llm("Hey, what's up?")

"Hey! Not much, just here and ready to chat. What's going on with you?"

In [22]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [23]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [24]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [26]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [27]:
documents[230]

{'id': 'c180431de3',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 4: Analytics Engineering with dbt',
 'question': 'dbt + BigQuery: "Dataset was not found in location" / region mismatch errors',
 'answer': 'This is the single most common dbt+BigQuery problem in the course. The various "404 Not found: Dataset was not found in location X" errors all come down to the same root cause: a region mismatch between\n\n- the source dataset (e.g. `trips_data_all`),\n- the dbt-managed dev/prod dataset (e.g. `dbt_<initial><lastname>` or `prod`), and\n- the connection\'s configured location in dbt Cloud or `profiles.yml`.\n\nBigQuery cannot read and write across regions in a single query, and dbt creates new datasets in whatever location its connection is configured to use. If those don\'t match your source data\'s region, you\'ll see one of:\n\n```\n404 Not found: Dataset <project>:<schema> was not found in location <region>\nAccess Denied: ... or perhaps it does not exist in locatio